In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from haversine import haversine_vector, Unit
import itertools
import requests, zipfile, io

**postal code-postal code distance**

In [30]:
# data from BokanyiE
irsz_data = pd.read_csv("../data/iranyitoszamok_BE.csv")
irsz_data["irsz"] = irsz_data["irsz"].astype(int)
irsz_data = irsz_data[["irsz", "lon", "lat"]].drop_duplicates(subset="irsz")

In [34]:
# dataframe w/ full combinations
pcodes = irsz_data["irsz"].unique()
full_comb = pd.MultiIndex.from_product(
    [pcodes, pcodes], names=["pcode1", "pcode2"]
)
full_comb = pd.DataFrame(index=full_comb).reset_index().sort_values(by=["pcode1", "pcode2"])
print(full_comb.shape)

(9278116, 2)


In [35]:
# add coords
full_comb = pd.merge(
    full_comb,
    irsz_data[["irsz", "lon", "lat"]].drop_duplicates(),
    left_on="pcode1",
    right_on="irsz",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    irsz_data[["irsz", "lon", "lat"]].drop_duplicates(),
    left_on="pcode2",
    right_on="irsz",
    how="left",
    suffixes=["1", "2"]
)
full_comb.drop(columns=["pcode1", "pcode2"], inplace=True)
print(full_comb.shape)

(9278116, 6)


In [37]:
# distance calculation
full_comb["coords1"] = list(
    zip(full_comb["lat1"], full_comb["lon1"])
)
full_comb["coords2"] = list(
    zip(full_comb["lat2"], full_comb["lon2"])
)
full_comb["distance"] = haversine_vector(
    full_comb["coords1"].tolist(), full_comb["coords2"].tolist()
)

In [49]:
# export
full_comb = full_comb[["irsz1", "irsz2", "distance"]]
full_comb["distance"] = full_comb["distance"].astype(int)
export_full_comb = full_comb[full_comb["irsz1"] < full_comb["irsz2"]]
export_full_comb.to_csv("../outputs/pcode_pcode_distance.csv", index=False, sep=";")

**city-city distance**

In [77]:
def eov_dist(eov_x1, eov_y1, eov_x2, eov_y2):
    """distance between eov coords (in km)"""
    x = np.array([eov_x1, eov_x2])
    y = np.array([eov_y1, eov_y2])
    # first derivative
    dx1 = np.diff(x,1)
    dy1 = np.diff(y,1)
    # distances between points
    d = np.sqrt(dx1**2+dy1**2)
    return (d[0]/1000).astype(int)

In [78]:
# data from BokanyiE
set_df = pd.read_csv("../data/settlements.csv")
set_df = set_df[["NAME", "eov_x", "eov_y"]].drop_duplicates(subset="NAME")

In [79]:
# dataframe w/ full combinations
set_names = set_df["NAME"].unique()
full_comb = pd.MultiIndex.from_product(
    [set_names, set_names], names=["set1", "set2"]
)
full_comb = pd.DataFrame(index=full_comb).reset_index().sort_values(by=["set1", "set2"])
print(full_comb.shape)

(9960336, 2)


In [80]:
# add coords
full_comb = pd.merge(
    full_comb,
    set_df[["NAME", "eov_x", "eov_y"]].drop_duplicates(),
    left_on="set1",
    right_on="NAME",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    set_df[["NAME", "eov_x", "eov_y"]].drop_duplicates(),
    left_on="set2",
    right_on="NAME",
    how="left",
    suffixes=["1", "2"]
)
full_comb.drop(columns=["NAME1", "NAME2"], inplace=True)
print(full_comb.shape)

(9960336, 6)


In [81]:
# distance calculation
full_comb["distance"] = full_comb.apply(lambda r: eov_dist(r["eov_x1"], r["eov_y1"], r["eov_x2"], r["eov_y2"]), axis=1)

In [88]:
# export
full_comb = full_comb[["set1", "set2", "distance"]]
full_comb["distance"] = full_comb["distance"].astype(int)
export_full_comb = full_comb[full_comb["set1"] < full_comb["set2"]]
export_full_comb.to_csv("../outputs/settlement_settlement_distance.csv", index=False, sep=";")